# Lab 6b: Graph Learning and Embeddings

**Course:** IS353 - Graph and Network Analysis with Applications, UIT, VNUHCM  
**Lecturer:** Dr. Tran Hung Nghiep, 2025  

---

**Goals:**
- Train **Node2Vec** embeddings via random walks on a toy graph; visualize.
- Train a **GCN** on a feature-rich citation network (Cora); evaluate node classification.
- Compare **shallow vs deep** approaches; inspect embeddings with t-SNE.
- (Optional) Try **GraphSAGE/GAT** and a **larger dataset** block.


## 0. Environment Setup

In [ ]:
# Install if needed (uncomment on your machine/Colab)
# !pip install torch torchvision torchaudio

!pip install torch-geometric
# !pip install torch-scatter torch-sparse  # bug on colab -> install on local.
# !pip install torch-cluster torch-spline-conv  # bug on colab -> install on local.

# !pip install networkx matplotlib scikit-learn



In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub, Planetoid
from torch_geometric.nn import Node2Vec, GCNConv
from torch_geometric.utils import to_networkx

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

plt.rcParams["figure.figsize"] = (7,5)
plt.rcParams["axes.grid"] = True

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
device.type

## 1. Datasets

In [ ]:
karate = KarateClub()
data_karate = karate[0].to(device.type)

cora = Planetoid(root='/tmp/Cora', name='Cora')
data_cora = cora[0].to(device.type)

print("Karate:", data_karate)
print("Cora:", data_cora)

## 2. Node2Vec: Shallow Random-Walk Embeddings (Karate Club)

In [ ]:
n2v = Node2Vec(
    data_karate.edge_index,
    embedding_dim=64,
    walk_length=10,
    context_size=5,
    walks_per_node=5,
    num_negative_samples=5,
    p=1.0, q=1.0,
    sparse=True
).to(device.type)

loader = n2v.loader(batch_size=128, shuffle=True)
optimizer = torch.optim.SparseAdam(n2v.parameters(), lr=0.01)

def train_node2vec(epochs=50):
    n2v.train()
    for epoch in range(1, epochs+1):
        loss = n2v.train_epoch(optimizer, loader)
        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | Loss {loss:.4f}")
    return n2v()

emb_karate = train_node2vec(epochs=50).detach().to('cpu')
emb_karate[:3]

## 3. Inspect Learned Embeddings (Node2Vec)

In [ ]:
Gk = to_networkx(data_karate, to_undirected=True)
pos = nx.spring_layout(Gk, seed=42)

plt.figure()
nx.draw(Gk, pos, with_labels=True, node_size=400)
plt.title("Karate Club Graph (structure)")
plt.show()

tsne2d = TSNE(n_components=2, init='random', learning_rate='auto').fit_transform(emb_karate.numpy())
plt.figure()
plt.scatter(tsne2d[:,0], tsne2d[:,1])
plt.title("Node2Vec Embeddings (t-SNE, Karate)")
plt.show()

## 4. Graph Convolutional Network: Node Classification (Cora)

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, data):
        """
        GCNConv layer computes over the whole graph (or subgraph), automatically goes over all nodes and its neighbors like in CNN.
        Input x is (#nodes, #features), edge_index provides neighborhood info, output x (#nodes, #features) also covers the whole graph.
        Stacking allows multi-round message passing in 1 forward pass (useful for computing node embeddings from raw features & connections).
        This is different than shallow embedding models where we compute new node embeddings stochastically via gradient descent with single-round message passing within each epoch, and simulate multi-round message passing after multiple epochs using newly updated node embeddings after each epoch.
        """
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model = GCN(cora.num_node_features, 16, cora.num_classes).to(device.type)
data = data_cora
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def train_gcn():
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss)

@torch.no_grad()
def test_gcn():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / int(mask.sum())
        accs.append(acc)
    return accs

for epoch in range(1, 20001):
    loss = train_gcn()
    if epoch % 20 == 0:
        train_acc, val_acc, test_acc = test_gcn()
        print(f"Epoch {epoch:03d} | Loss {loss:.3f} | Val {val_acc:.3f} | Test {test_acc:.3f}")

## 5. Inspect Learned Embeddings (GCN)

In [ ]:
@torch.no_grad()
def extract_embeddings(model, data):
    model.eval()
    x, edge_index = data.x, data.edge_index
    h = model.conv1(x, edge_index)
    h = F.relu(h)
    return h

emb_cora = extract_embeddings(model, data)
emb2d = TSNE(n_components=2, init='random', learning_rate='auto').fit_transform(emb_cora.detach().to('cpu').numpy())

plt.figure()
plt.scatter(emb2d[:,0], emb2d[:,1])
plt.title("GCN Hidden Embeddings (t-SNE, Cora)")
plt.show()

## 6. Questions

- Compare Node2Vec vs. GCN embeddings (t-SNE plots). Do you see clearer clusters with GCN on Cora?
- Run GCN longer until convergence. Report final **test accuracy** for GCN. Does increasing hidden units/epochs help?
- Change Node2Vec `p,q` to bias BFS/DFS. Describe effect on the embedding plot.
- (Optional) Build a k-NN graph using cosine similarity on embeddings and compare to original edges.

## 7. Optional Extensions

- Replace `GCNConv` with `SAGEConv` or `GATConv` and compare results.
- Use a larger dataset (e.g., SNAP Facebook) and craft simple node features (degree, clustering).
- Try **link prediction**: construct edge features from embeddings (Hadamard, L1, L2) and train a logistic regression.
- Hyperparameter exploration: embedding_dim, dropout, walk_length/context_size, learning rate.